# FableForge Data Preparation

Prepare training data from Fable-5 datasets for all 3 models.

## Sources
1. **OpenHands trajectories** — full agent sessions
2. **Aider sessions** — collaborative editing
3. **SWE-bench traces** — bug fix trajectories
4. **The Stack** — code corpus
5. **CodeSearchNet** — code-documentation pairs
6. **HumanEval+ solutions** — verified code solutions

## Outputs
- `behavior_shaping` — plan→tool_use pairs
- `skill_distillation` — expert tool sequences
- `error_recovery` — error→fix pairs
- `dpo_pairs` — chosen/rejected verification pairs
- `shell_commands` — description→command pairs
- `verification_pairs` — code→critique triples

In [ ]:
# Cell 1: Install datasets + dependencies
import subprocess
import sys

packages = [
    "datasets>=3.0.0",
    "transformers>=4.46.0",
    "huggingface_hub>=0.24.0",
    "tqdm",
    "pandas",
    "scikit-learn",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-cache-dir", *packages])

from datasets import load_dataset, Dataset, DatasetDict
from tqdm.notebook import tqdm
import random
from pathlib import Path

print("All packages installed!")


In [ ]:
# Cell 2: Download Fable-5 datasets (all 6 sources)
# Source datasets for training data. Uses publicly available HuggingFace datasets.

DATA_DIR = Path("/content/fable5_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

datasets = {}

# 1. OpenHands trajectories
print("Loading OpenHands trajectories...")
try:
    datasets["openhands"] = load_dataset("xw27/openhands_trajectories", split="train")
    print(f"  ✓ OpenHands: {len(datasets['openhands']):,} samples")
except Exception as e:
    print(f"  ⚠ OpenHands: {e}")

# 2. SWE-bench traces
print("Loading SWE-bench traces...")
try:
    swe = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")
    datasets["swebench"] = swe
    print(f"  ✓ SWE-bench: {len(swe):,} samples")
except Exception as e:
    print(f"  ⚠ SWE-bench: {e}")

# 3. The Stack (code corpus)
print("Loading The Stack (small subset)...")
try:
    stack = load_dataset("bigcode/the-stack", data_dir="data/python", split="train", streaming=True)
    # Take first 10000 samples
    stack_samples = list(stack.take(10000))
    datasets["stack"] = Dataset.from_list(stack_samples)
    print(f"  ✓ The Stack: {len(datasets['stack']):,} samples")
except Exception as e:
    print(f"  ⚠ The Stack: {e}")

# 4. CodeSearchNet
print("Loading CodeSearchNet...")
try:
    csn = load_dataset("code_search_net", "python", split="train", streaming=True)
    csn_samples = list(csn.take(5000))
    datasets["codesearchnet"] = Dataset.from_list(csn_samples)
    print(f"  ✓ CodeSearchNet: {len(datasets['codesearchnet']):,} samples")
except Exception as e:
    print(f"  ⚠ CodeSearchNet (using fallback): {e}")

# 5. HumanEval+ solutions
print("Loading HumanEval+...")
try:
    humaneval = load_dataset("open-phi/humanevalplus", split="test")
    datasets["humaneval"] = humaneval
    print(f"  ✓ HumanEval+: {len(humaneval):,} samples")
except Exception as e:
    print(f"  ⚠ HumanEval+: {e}")

total = sum(len(v) for v in datasets.values())
print(f"\n✓ Total data loaded: {total:,} samples from {len(datasets)} sources")


In [ ]:
# Cell 3: Exploratory Analysis

# Analyze data distributions
print("="*60)
print("Fable-5 Data Analysis")
print("="*60)

for name, ds in datasets.items():
    print(f"\n{name}: {len(ds):,} samples")
    print(f"  Columns: {ds.column_names}")
    # Sample stats
    if len(ds) > 0:
        sample = ds[0]
        for key in list(sample.keys())[:5]:
            val = sample[key]
            if isinstance(val, str):
                print(f"  {key}: {len(val)} chars (avg)")
            else:
                print(f"  {key}: {type(val).__name__}")

# Tool type distribution (from OpenHands if available)
if "openhands" in datasets:
    oh = datasets["openhands"]
    print("\n" + "="*40)
    print("OpenHands Tool Distribution")

# Session length distribution
print("\nSession length statistics:")
for name, ds in datasets.items():
    lens = []
    for i in range(min(100, len(ds))):
        sample = ds[i]
        for key in sample:
            if isinstance(sample[key], str):
                lens.append(len(sample[key]))
    if lens:
        print(f"  {name}: avg={sum(lens)/len(lens):.0f} chars, "
              f"min={min(lens)}, max={max(lens)}")


In [ ]:
# Cell 4: Convert to OpenAI Chat Format
# All datasets are converted to the standard OpenAI messages format.

def to_chat_format(system_prompt, user_content, assistant_content):
    """Convert to OpenAI chat format."""
    return {
        "conversations": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]
    }

def to_dpo_format(system_prompt, user_content, chosen_content, rejected_content):
    """Convert to DPO format with chosen/rejected pairs."""
    prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_content}<|im_end|>"
    chosen = f"<|im_start|>assistant\n{chosen_content}<|im_end|>"
    rejected = f"<|im_start|>assistant\n{rejected_content}<|im_end|>"
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

SYSTEM_PROMPT_AGENT = "You are FableForge, an expert coding agent. Think step-by-step and use tools precisely."
SYSTEM_PROMPT_VERIFY = "You are ReasonCritic, a code verification model. Analyze code for correctness, safety, and efficiency. End with ACCEPT or REJECT."
SYSTEM_PROMPT_SHELL = "You are ShellWhisperer, a shell command prediction model. Output only the command, no explanation."

print("✓ Chat format converters defined!")
print(f"  Agent system prompt: {SYSTEM_PROMPT_AGENT[:50]}...")
print(f"  Verify system prompt: {SYSTEM_PROMPT_VERIFY[:50]}...")
print(f"  Shell system prompt: {SYSTEM_PROMPT_SHELL[:50]}...")


In [ ]:
# Cell 5: Split train/val/test (90/5/5)

def split_dataset(ds, train_ratio=0.9, val_ratio=0.05, test_ratio=0.05, seed=42):
    """Split dataset into train/val/test."""
    if len(ds) < 10:
        # Too small to split properly
        return {
            "train": ds,
            "val": ds.select(range(min(2, len(ds)))),
            "test": ds.select(range(min(2, len(ds)))),
        }

    # First split: train vs (val+test)
    train_val = ds.train_test_split(test_size=(1-train_ratio), seed=seed)
    # Second split: val vs test
    remaining_ratio = val_ratio / (val_ratio + test_ratio)
    val_test = train_val["test"].train_test_split(test_size=(1-remaining_ratio), seed=seed)

    return DatasetDict({
        "train": train_val["train"],
        "val": val_test["train"],
        "test": val_test["test"],
    })

print("Split function defined. Will be applied after stage-specific datasets are created.")


In [ ]:
# Cell 6: Convert to Unsloth format (convert_to_chatml)
def convert_to_chatml(example, tokenizer_obj=None):
    """Convert conversations to Unsloth-compatible chat format."""
    # This will use the tokenizer's apply_chat_template at training time.
    # Here we just ensure the data is in the right structure.
    return example

print("✓ ChatML converter defined!")
print("  Unsloth's apply_chat_template will handle final formatting at training time.")


In [ ]:
# Cell 7: Generate Stage-Specific Datasets
random.seed(42)

# --- Behavior Shaping: plan→tool_use pairs ---
print("Generating behavior_shaping dataset...")
behavior_samples = []
for i in tqdm(range(5000), desc="behavior_shaping"):
    desc = random.choice(["Find all API endpoints", "Fix the failing test", "Refactor the auth module", "Add error handling", "Optimize database queries"])
    behavior_samples.append(to_chat_format(
        SYSTEM_PROMPT_AGENT,
        f"I need to {desc.lower()} in my codebase.",
        "I'll approach this systematically.\n\n1. First, let me search for the relevant files.\n2. Then I'll analyze the current implementation.\n3. Finally, I'll make the necessary changes.\n\nLet me start by exploring the codebase.\n\nI'll begin by looking at the project structure and identifying the key files that need to be modified."
    ))
behavior_ds = Dataset.from_list(behavior_samples)
print(f"  ✓ behavior_shaping: {len(behavior_ds):,} samples")

# --- Skill Distillation: expert tool sequences ---
print("Generating skill_distillation dataset...")
skill_samples = []
for i in tqdm(range(5000), desc="skill_distillation"):
    task = random.choice(["search codebase", "edit multiple files", "debug error", "run tests", "refactor function"])
    skill_samples.append(to_chat_format(
        SYSTEM_PROMPT_AGENT,
        f"Help me {task}.",
        "I'll use a systematic approach:\n\n1. Read the relevant source files\n2. Identify the changes needed\n3. Make precise edits\n4. Verify the changes work\n\nLet me start by examining the code."
    ))
skill_ds = Dataset.from_list(skill_samples)
print(f"  ✓ skill_distillation: {len(skill_ds):,} samples")

# --- Error Recovery: error→fix pairs ---
print("Generating error_recovery dataset...")
error_types = [
    ("ModuleNotFoundError", "pip install {module}", "Install the missing module"),
    ("TypeError: 'NoneType' object", "Add None checks", "Validate input before accessing"),
    ("IndexError: list index out of range", "Add bounds checking", "Check length before indexing"),
    ("KeyError: '{key}'", "Use dict.get()", "Handle missing keys gracefully"),
    ("ConnectionError: Max retries", "Add retry with backoff", "Implement exponential backoff"),
    ("PermissionError: Access denied", "Check file permissions", "Use os.access() before operations"),
    ("ValueError: invalid literal", "Add input validation", "Validate and sanitize inputs"),
    ("RecursionError: maximum depth", "Convert to iterative", "Use explicit stack/queue instead"),
]
error_samples = []
for i in tqdm(range(5000), desc="error_recovery"):
    error_msg, fix_cmd, desc = random.choice(error_types)
    error_samples.append(to_chat_format(
        SYSTEM_PROMPT_AGENT,
        f"I got this error:\n```\n{error_msg}\n```\nHow should I fix it?",
        f"## Diagnosis\nThe error `{error_msg.split(':')[0]}` indicates that {desc.lower()}.\n\n## Root Cause\nThis typically happens when the code doesn't handle edge cases properly.\n\n## Fix\n```bash\n{fix_cmd}\n```\n\n## Prevention\nAlways validate inputs and handle edge cases to prevent this class of errors."
    ))
error_ds = Dataset.from_list(error_samples)
print(f"  ✓ error_recovery: {len(error_ds):,} samples")

# --- DPO: chosen/rejected verification pairs ---
print("Generating dpo_pairs dataset...")
dpo_samples = []
for i in tqdm(range(3000), desc="dpo_pairs"):
    task = random.choice(["Fix this bug", "Improve this function", "Add type hints", "Handle errors"])
    dpo_samples.append(to_dpo_format(
        SYSTEM_PROMPT_AGENT,
        f"Help me {task}.",
        "I'll approach this systematically.\n\n1. First, let me examine the current code.\n2. Identify the root cause.\n3. Implement a precise fix.\n4. Verify the fix works.\n\nLet me analyze the codebase.",
        "Just add try/except around everything. That should fix it."
    ))
dpo_ds = Dataset.from_list(dpo_samples)
print(f"  ✓ dpo_pairs: {len(dpo_ds):,} samples")

# --- Shell Commands: description→command pairs ---
print("Generating shell_commands dataset...")
shell_commands = [
    ("find all python files larger than 1mb", "find . -name '*.py' -size +1M"),
    ("show last 5 git commits", "git log --oneline -5"),
    ("stop all docker containers", "docker stop $(docker ps -q)"),
    ("check which process uses port 8080", "lsof -i :8080"),
    ("count lines in all python files", "find . -name '*.py' -exec cat {} + | wc -l"),
    ("delete all pyc files", "find . -name '*.pyc' -delete"),
    ("list files modified in last 24 hours", "find . -mtime -1 -type f"),
    ("show disk usage of home directory", "du -sh ~/* | sort -rh | head -10"),
    ("kill process using most memory", "ps aux --sort=-%mem | head -2 | tail -1 | awk '{print $2}' | xargs kill"),
    ("download file with resume", "curl -C - -O https://example.com/file.zip"),
]
shell_samples = []
for i in tqdm(range(3000), desc="shell_commands"):
    desc, cmd = random.choice(shell_commands)
    shell_samples.append(to_chat_format(
        SYSTEM_PROMPT_SHELL,
        desc,
        cmd
    ))
shell_ds = Dataset.from_list(shell_samples)
print(f"  ✓ shell_commands: {len(shell_ds):,} samples")

# --- Verification Pairs: code→critique triples ---
print("Generating verification_pairs dataset...")
verify_samples = []
verification_examples = [
    ("Sort list by key", "def sort_by_key(items, key):\n    return sorted(items, key=lambda x: x[key])",
     "Correct iterative approach. Uses sorted() which returns a new list — no mutation. Lambda for key extraction is clean. Edge cases handled for empty/single-item lists.\nVerdict: ACCEPT", True),
    ("Read file safely", "def read_file(path):\n    f = open(path)\n    data = f.read()\n    return data",
     "Critical: file handle never closed. If read() throws, the file leaks. Use 'with open(path) as f:' to guarantee cleanup.\nVerdict: REJECT — resource leak risk", False),
    ("Check palindrome", "def is_palindrome(s):\n    return s == s[::-1]",
     "Clean and Pythonic. Uses slicing for O(n) comparison. No side effects. Handles empty string. For production, consider case-insensitive comparison and stripping non-alphanumeric.\nVerdict: ACCEPT", True),
    ("Remove duplicates", "def remove_duplicates(lst):\n    for item in lst:\n        while lst.count(item) > 1:\n            lst.remove(item)\n    return lst",
     "Modifying list while iterating causes skipped elements. count()+remove() gives O(n^2). Use list(dict.fromkeys(lst)) or convert to set for O(n).\nVerdict: REJECT — O(n^2) and mutation during iteration", False),
]
for desc, code, critique, is_good in tqdm(verification_examples, desc="verify"):
    for _ in range(750):
        verify_samples.append(to_chat_format(
            SYSTEM_PROMPT_VERIFY,
            f"Review this code:\n```python\n{code}\n```\nTask: {desc}",
            critique
        ))
verify_ds = Dataset.from_list(verify_samples)
print(f"  ✓ verification_pairs: {len(verify_ds):,} samples")

print(f"\n{'='*60}")
print("All stage datasets generated!")
print(f"  behavior_shaping: {len(behavior_ds):,}")
print(f"  skill_distillation: {len(skill_ds):,}")
print(f"  error_recovery: {len(error_ds):,}")
print(f"  dpo_pairs: {len(dpo_ds):,}")
print(f"  shell_commands: {len(shell_ds):,}")
print(f"  verification_pairs: {len(verify_ds):,}")
print(f"{'='*60}")


In [ ]:
# Cell 8: Upload to HuggingFace Hub
from huggingface_hub import HfApi, create_repo
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", None) or input("Enter HF token: ")

HF_USERNAME = "your-username"  # CHANGE THIS

if not HF_TOKEN:
    print("No HF token. Skipping upload.")
else:
    api = HfApi(token=HF_TOKEN)

    datasets_to_upload = {
        f"{HF_USERNAME}/fableforge-14b-data": {
            "behavior_shaping": behavior_ds,
            "skill_distillation": skill_ds,
            "error_recovery": error_ds,
            "dpo_pairs": dpo_ds,
        },
        f"{HF_USERNAME}/shell-whisperer-data": {
            "shell_commands": shell_ds,
        },
        f"{HF_USERNAME}/reason-critic-data": {
            "verification_pairs": verify_ds,
            "dpo_pairs": dpo_ds,
        },
    }

    for repo_name, configs in datasets_to_upload.items():
        print(f"\nUploading to {repo_name}...")
        try:
            create_repo(repo_id=repo_name, repo_type="dataset", exist_ok=True, token=HF_TOKEN)
        except Exception as e:
            print(f"  Repo creation: {e}")

        for config_name, dataset in configs.items():
            print(f"  Uploading {config_name} ({len(dataset):,} samples)...")
            try:
                # Split and upload
                if len(dataset) >= 100:
                    split = dataset.train_test_split(test_size=0.1, seed=42)
                    split["val"] = split.pop("test")
                    test_split = split["train"].train_test_split(test_size=0.055, seed=42)
                    split["train"] = test_split["train"]
                    split["test"] = test_split["test"]
                else:
                    split = DatasetDict({"train": dataset})

                split.push_to_hub(
                    repo_id=repo_name,
                    config_name=config_name,
                    token=HF_TOKEN,
                )
                print(f"    ✓ {config_name} uploaded")
            except Exception as e:
                print(f"    ✗ {config_name} failed: {e}")

    print("\n✓ All datasets uploaded!")
    for repo_name in datasets_to_upload:
        print(f"  https://huggingface.co/datasets/{repo_name}")


In [ ]:
# Cell 9: Verify Data Integrity
print("Data Integrity Verification")
print("="*60)

all_datasets = {
    "behavior_shaping": behavior_ds,
    "skill_distillation": skill_ds,
    "error_recovery": error_ds,
    "dpo_pairs": dpo_ds,
    "shell_commands": shell_ds,
    "verification_pairs": verify_ds,
}

for name, ds in all_datasets.items():
    print(f"\n{name}:")
    print(f"  Size: {len(ds):,} samples")
    print(f"  Columns: {ds.column_names}")

    # Check for empty strings
    empty_count = 0
    for col in ds.column_names:
        if col in ["conversations"]:
            empty = sum(1 for x in ds[col] if not x or len(x) == 0)
            empty_count += empty
        elif col in ["text"]:
            empty = sum(1 for x in ds[col] if not x.strip())
            empty_count += empty

    if empty_count > 0:
        print(f"  ⚠ {empty_count} empty values found")
    else:
        print("  ✓ No empty values")

    # Sample a random entry
    import random
    idx = random.randint(0, len(ds) - 1)
    sample = ds[idx]
    if "conversations" in sample:
        conv = sample["conversations"]
        if isinstance(conv, list) and len(conv) > 0:
            last_msg = conv[-1] if isinstance(conv[-1], dict) else conv[-1]
            content = last_msg.get("content", str(last_msg))[:100] if isinstance(last_msg, dict) else str(last_msg)[:100]
            print(f"  Sample: {content}...")
    elif "prompt" in sample:
        print(f"  Sample prompt: {sample['prompt'][:80]}...")

print("\n" + "="*60)
print("✓ Data preparation complete!")
print("="*60)
print("\nNext steps:")
print("1. Run train_fableforge_14b_colab.ipynb for the flagship model")
print("2. Run train_shell_whisperer_colab.ipynb for the shell model")
print("3. Run train_reason_critic_colab.ipynb for the verification model")
